# 🖼️ Usecase 6: Multimodal GCS Offloading, Object Tables & Looker BI Integration

This notebook demonstrates advanced enterprise capabilities:
1. **Multimodal GCS Offloading**: How `BigQueryLoggerConfig(gcs_bucket_name="japac-pso-agent-analytics")` offloads large text and multimodal media (images, audio) to Google Cloud Storage and links them via BigQuery ObjectRef tables (`connection_id`).
2. **Looker BI Marketplace & Open-Source Blocks**: Architecture for connecting Looker Business Intelligence to the 24 unnested SQL views (`v_llm_request`, `v_tool_completed`, etc.).

```mermaid
flowchart LR
    A[Agent Large Text / Image / Audio] -->|Offload| B[(GCS Bucket: japac-pso-agent-analytics)]
    A -->|Write Metadata & object_ref| C[(BigQuery agent_events)]
    B <-->|BigQuery Cloud Resource Connection| D[BigQuery ML Object Tables]
    C --> E[Looker Marketplace Pack]
    C --> F[Looker Open-Source LookML Block]
```

In [1]:
import os
from google.auth import default
from google.cloud import bigquery

credentials, _ = default()
PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"

bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print("✅ Connected to BigQuery for Multimodal & Looker BI Analysis.")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Connected to BigQuery for Multimodal & Looker BI Analysis.


## 1) Multimodal GCS Offloading & ObjectRef Schema

When `gcs_bucket_name="japac-pso-agent-analytics"` is configured in `BigQueryLoggerConfig`, the plugin offloads payloads exceeding `max_content_length` (or binary media) to GCS and records a JSON reference in `content_parts`.

```json
{
  "event_type": "LLM_REQUEST",
  "content_parts": [
    {
      "part_index": 1,
      "mime_type": "text/plain",
      "storage_mode": "GCS_REFERENCE",
      "text": "[OFFLOADED TO GCS]",
      "object_ref": {
        "uri": "gs://japac-pso-agent-analytics/2026-07-27/session_01/part_1.txt",
        "authorizer": "asia-southeast1.bqml_connection"
      }
    }
  ]
}
```

In [2]:
query_gcs_refs = f"""
    SELECT
        timestamp,
        event_type,
        part.mime_type,
        part.storage_mode,
        part.object_ref.uri AS gcs_uri
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`,
    UNNEST(JSON_QUERY_ARRAY(content_parts)) AS part_json,
    UNNEST([STRUCT(
        JSON_VALUE(part_json, '$.mime_type') AS mime_type,
        JSON_VALUE(part_json, '$.storage_mode') AS storage_mode,
        STRUCT(JSON_VALUE(part_json, '$.object_ref.uri') AS uri) AS object_ref
    )]) AS part
    WHERE part.storage_mode = 'GCS_REFERENCE'
    LIMIT 10
"""
try:
    df_gcs = bq_client.query(query_gcs_refs).to_dataframe()
    display(df_gcs)
except Exception as e:
    print(f"GCS ObjectRef Query Note: {e} (No GCS offloaded rows recorded yet)")


GCS ObjectRef Query Note: 400 No matching signature for function JSON_QUERY_ARRAY
  Argument types: ARRAY<STRUCT<mime_type STRING, uri STRING, object_ref STRUCT<uri STRING, version STRING, authorizer STRING, ...>, ...>>
  Signature: JSON_QUERY_ARRAY(STRING, [STRING])
    Argument 1: Unable to coerce type ARRAY<STRUCT<mime_type STRING, uri STRING, object_ref STRUCT<uri STRING, version STRING, authorizer STRING, ...>, ...>> to expected type STRING
  Signature: JSON_QUERY_ARRAY(JSON, [STRING])
    Argument 1: Unable to coerce type ARRAY<STRUCT<mime_type STRING, uri STRING, object_ref STRUCT<uri STRING, version STRING, authorizer STRING, ...>, ...>> to expected type JSON at [9:12]; reason: invalidQuery, location: query, message: No matching signature for function JSON_QUERY_ARRAY
  Argument types: ARRAY<STRUCT<mime_type STRING, uri STRING, object_ref STRUCT<uri STRING, version STRING, authorizer STRING, ...>, ...>>
  Signature: JSON_QUERY_ARRAY(STRING, [STRING])
    Argument 1: Unable to c

## 2) Looker BI Marketplace Pack & Open-Source LookML Block Reference

BigQuery Agent Analytics is designed for zero-reinstrumentation integration with Google Cloud Looker:
- **Looker Marketplace Detail**: [https://marketplace.looker.com/marketplace/detail/agent_analytics](https://marketplace.looker.com/marketplace/detail/agent_analytics)
- **Looker Open-Source GitHub Repository**: [https://github.com/looker-open-source/agent-analytics-block](https://github.com/looker-open-source/agent-analytics-block)

### How Looker Connects to the 24 Unnested SQL Views (`v_*`):
1. **`v_llm_request` & `v_llm_response`**: Powers executive dashboards for FinOps token costs and model usage distribution.
2. **`v_tool_completed`**: Powers tool reliability and latency leaderboards.
3. **`v_user_message_received` & `v_agent_response`**: Powers conversational analytics and customer intent reporting.
4. **`v_agent_transfer` & `v_agent_state_checkpoint`**: Powers multi-agent ADK 2.0 graph visualization.